In [14]:
import os
from warnings import filters

import jax.numpy as jnp
import jax
from caskade import Param, forward
import numpy as np
from cosmographi.cosmology import Cosmology
from cosmographi.source.base import TransientSource
from cosmographi.utils import flux
from cosmographi.utils.constants import Mpc_to_cm
from typing import Any

# V. 1.0: No parameters
The following code shows the use of an AGN source that has a constant luminosity density for testing purposes.

In [15]:
class AGNSourcev1_0(TransientSource):
    """
    An AGN source model.

    Note: This is a simplified model that should not be used yet. 

    Parameters
    ---------- 
    name: str. 
        Optional. Name of the source.
    """
    cosmology: Cosmology
    name: str

    def __init__(self, cosmology: Cosmology = None, name: str = None, **kwargs) -> None:
        super().__init__(cosmology=cosmology, name=name, **kwargs)
    
    def luminosity_density(self, t: jnp.ndarray) -> jnp.ndarray:
        """
        Compute the luminosity density at time t. 

        Parameters
        ----------
        t: jnp.ndarray. Time array in seconds. Either (N,) or (N,1)

        Returns
        -------
        luminosity_density: jnp.ndarray. Luminosity density in erg/s/Hz.
        """
        if t.ndim == 1:
            t = t[:, None]
        density = jnp.full((len(t),1), 0.04)
        return jnp.concatenate((t, density), axis=1)
        


In [16]:
agn = AGNSourcev1_0(name="AGN1")
print(agn.luminosity_density(jnp.array([0.0, 0.5, 3.0, 5.0])))

[[0.   0.04]
 [0.5  0.04]
 [3.   0.04]
 [5.   0.04]]


# V.1.1: One parameter, luminosity if a function of time and said parameter
The following code shows an AGN whose luminosity is a function of time and a parameter of the AGN

In [20]:
from astropy.time import Time

class AGNSourcev1_1(TransientSource):
    """
    An AGN source model.

    Note: This is a simplified model that should not be used yet. 

    Parameters
    ---------- 
    name: str. 
        Optional. Name of the source.
    t0: 
        Param. Date of the initial obsevation in MJD format.
    lum_at_t0:
        Param. Luminosity density at t0 in erg/s/Hz.

    """
    cosmology: Cosmology
    name: str
    t0: Time
    lum_at_t0: Param

    def __init__(self, cosmology: Cosmology = None, name: str = None, t0: float = None, lum_at_t0: float = None, **kwargs) -> None:
        super().__init__(cosmology=cosmology, name=name, **kwargs)
        self.t0 = Param("t0", t0, shape=(), description="Date of the initial observation in MJD format")
        self.lum_at_t0 = Param("lum_at_t0", lum_at_t0, shape=(), description="Luminosity density at t0 in erg/s/Hz")

    def luminosity_density(self, t: jnp.ndarray) -> jnp.ndarray:
        """
        Compute the luminosity density at time t. 

        Parameters
        ----------
        t: jnp.ndarray. Time array in seconds. Either (N,) or (N,1)

        Returns
        -------
        luminosity_density: jnp.ndarray. Luminosity density in erg/s/Hz.
        """
        if t.ndim == 1:
            t = t[:, None]
        density = jnp.full((len(t),1), (self.lum_at_t0.value + 0.01 * (t - self.t0.value)))
        print("t:", t)
        print("density:", density)
        return jnp.concatenate((t, density), axis=1)

In [22]:
agn = AGNSourcev1_1(name="AGN2", t0=Time("2023-01-01T00:00:00").mjd, lum_at_t0=0.04)
print(agn.luminosity_density(jnp.array([Time("2023-01-01T00:00:00").mjd, Time("2023-01-01T00:00:00").mjd + 0.5, Time("2023-01-01T00:00:00").mjd + 3.0, Time("2023-01-01T00:00:00").mjd + 5.0])))

t: [[59945. ]
 [59945.5]
 [59948. ]
 [59950. ]]
density: [[0.04 ]
 [0.045]
 [0.07 ]
 [0.09 ]]
[[5.99450e+04 4.00000e-02]
 [5.99455e+04 4.50000e-02]
 [5.99480e+04 7.00000e-02]
 [5.99500e+04 9.00000e-02]]


# V.1.2: One parameter, but infered through interpolation
Same as above, but the parameter is infered from some data points, which results in us using more of Caskade's functionality. 
The code below is used to generate the samples:

In [ ]:
t_i = Time("2026-01-01T00:00:00")
sample_times = jnp.array([t_i + 0.0, t_i + 0.5, t_i + 1.0, t_i + 1.5, t_i + 2.0, t_i + 2.5, t_i + 3.0, t_i + 3.5, t_i + 4.0, t_i + 4.5, 
                         t_i + 5.0, t_i + 5.5, t_i + 6.0])
true_lum_at_t0 = 0.04
samples = []
for time in sample_times:
    samples.append(np.random.normal(loc=true_lum_at_t0 - 0.01 * (time.mjd - t_i.mjd), scale = 0.005))
samples_jnp = jnp.array(samples)

In [ ]:
def infer_parameter(samples: jnp.ndarray, sample_times: jnp.ndarray) -> float:
    """
    Infer the parameter lum_at_t0 from the samples. 

    Parameters
    ----------
    samples: jnp.ndarray. Array of luminosity density samples in erg/s/Hz.
    sample_times: jnp.ndarray. Array of sample times in MJD format.

    Returns
    -------
    inferred_lum_at_t0: float. Inferred value of lum_at_t0.
    """
    # Fit a line to the samples to infer lum_at_t0
    




class AGNSourcev1_2(TransientSource):
    """
    An AGN source model.

    Note: This is a simplified model that should not be used yet. 

    Parameters
    ---------- 
    name: str. 
        Optional. Name of the source.
    t0: 
        Param. Date of the initial obsevation in MJD format.
    lum_at_t0:
        Param. Luminosity density at t0 in erg/s/Hz.

    """
    cosmology: Cosmology
    name: str
    t0: Param
    lum_at_t0: Param

    def __init__(self, cosmology: Cosmology = None, name: str = None, t0: float = None, lum_at_t0: float = None, **kwargs) -> None:
        super().__init__(cosmology=cosmology, name=name, **kwargs)
        self.t0 = Param("t0", t0, shape=(), description="Date of the initial observation in MJD format")
        self.lum_at_t0 = Param("lum_at_t0", lum_at_t0, shape=(), description="Luminosity density at t0 in erg/s/Hz")

    def luminosity_density(self, t: jnp.ndarray) -> jnp.ndarray:
        """
        Compute the luminosity density at time t. 

        Parameters
        ----------
        t: jnp.ndarray. Time array in seconds. Either (N,) or (N,1)

        Returns
        -------
        luminosity_density: jnp.ndarray. Luminosity density in erg/s/Hz.
        """
        if t.ndim == 1:
            t = t[:, None]
        density = jnp.full((len(t),1), (self.lum_at_t0.value + 0.01 * (t - self.t0.value)))
        print("t:", t)
        print("density:", density)
        return jnp.concatenate((t, density), axis=1)
    
    